# 03 - Phân tích nâng cao: PCA & Edge Detection

**Mục tiêu:** PCA eigenimages + Sobel/Canny edge detection, kèm thống kê so sánh.

**Dataset:** NWPU-RESISC45 (45 lớp, 256x256 pixel)

## 0. Setup

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from tqdm import tqdm
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 11, 'figure.dpi': 100})
sns.set_style("whitegrid")

TRAIN_DIR = r"D:\DataMining\Dataset\train"
classes = sorted(os.listdir(TRAIN_DIR))
print(f"Loaded: {len(classes)} lop")

In [ ]:
def load_sample(n_per_class=20, target_classes=None, seed=42):
    np.random.seed(seed)
    target = target_classes or classes
    samples = []
    for cls in target:
        paths = glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg"))
        chosen = np.random.choice(paths, min(n_per_class, len(paths)), replace=False)
        for p in chosen:
            img = cv2.imread(p)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            samples.append((img, cls))
    return samples

---
## 1. PCA đặc trưng ảnh

**H0:** Các lớp không phân tách được trên không gian PCA (PC1 mean không khác biệt).

**H1:** Ít nhất một cặp lớp phân tách được tren PC1.

In [ ]:
PCA_SIZE = 64
N_PCA_SAMPLE = 50

print(f"Loading {N_PCA_SAMPLE} anh/lop, resize {PCA_SIZE}x{PCA_SIZE}..")
pca_data = []
pca_labels = []

for cls in tqdm(classes, desc="Loading"):
    paths = glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg"))
    np.random.seed(42)
    chosen = np.random.choice(paths, min(N_PCA_SAMPLE, len(paths)), replace=False)
    for p in chosen:
        img = cv2.imread(p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = cv2.resize(img, (PCA_SIZE, PCA_SIZE))
        pca_data.append(img.ravel().astype(np.float32) / 255.0)
        pca_labels.append(cls)

X_pca = np.array(pca_data)
y_pca = np.array(pca_labels)
print(f"Ma tran PCA: {X_pca.shape} ({X_pca.shape[0]} anh x {X_pca.shape[1]} features)")

In [ ]:
pca = PCA(n_components=min(100, X_pca.shape[0]-1))
X_pca_transformed = pca.fit_transform(X_pca)

cum_var = np.cumsum(pca.explained_variance_ratio_)
n95 = np.argmax(cum_var >= 0.95) + 1
n99 = np.argmax(cum_var >= 0.99) + 1 if cum_var[-1] >= 0.99 else ">100"

print(f"Components cho 90% variance: {np.argmax(cum_var >= 0.90) + 1}")
print(f"Components cho 95% variance: {n95}")
print(f"Components cho 99% variance: {n99}")
print(f"Top-10 cumulative: {cum_var[:10].round(4)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 31), pca.explained_variance_ratio_[:30], color='#4C72B0', alpha=0.8)
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_title("Scree Plot (top 30)")

axes[1].plot(range(1, len(cum_var)+1), cum_var, color='#4C72B0', linewidth=2)
axes[1].axhline(0.95, color='red', linestyle='--', alpha=0.6, label='95%')
axes[1].axhline(0.90, color='orange', linestyle='--', alpha=0.6, label='90%')
axes[1].axvline(n95, color='red', linestyle=':', alpha=0.4)
axes[1].set_xlabel("So components")
axes[1].set_ylabel("Cumulative Explained Variance")
axes[1].set_title("Cumulative Explained Variance")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for idx, ax in enumerate(axes.ravel()):
    eigenimg = pca.components_[idx].reshape(PCA_SIZE, PCA_SIZE)
    ax.imshow(eigenimg, cmap='gray')
    ax.set_title(f"PC{idx+1} ({pca.explained_variance_ratio_[idx]*100:.1f}%)", fontsize=9)
    ax.axis('off')
plt.suptitle("Top 10 Eigenimages (Principal Components)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
mean_img = X_pca.mean(axis=0).reshape(PCA_SIZE, PCA_SIZE)
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(mean_img, cmap='gray')
ax.set_title("Mean Image (toan dataset)")
ax.axis('off')
plt.tight_layout()
plt.show()

### Chọn 10 lớp đại diện cho 2D projection

Từ kết quả PCA, chọn 10 lớp có PC1 mean trải đều nhất (5 thấp nhất + 5 cao nhất)
để minh họa sự phân tách giữa các lớp trên không gian PCA.

In [ ]:
pc1_per_class = {}
for cls in classes:
    mask = y_pca == cls
    pc1_per_class[cls] = X_pca_transformed[mask, 0].mean()

sorted_by_pc1 = sorted(pc1_per_class, key=pc1_per_class.get)
REPR_CLASSES = sorted_by_pc1[:5] + sorted_by_pc1[-5:]

print("10 lớp đại diện cho PCA 2D projection (5 PC1 thấp nhất + 5 cao nhất):")
for cls in REPR_CLASSES:
    print(f"  {cls}: PC1 mean = {pc1_per_class[cls]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.tab10(np.linspace(0, 1, len(REPR_CLASSES)))

for cls, color in zip(REPR_CLASSES, colors):
    mask = y_pca == cls
    ax.scatter(X_pca_transformed[mask, 0], X_pca_transformed[mask, 1],
               c=[color], label=cls, alpha=0.5, s=15)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("PCA 2D Projection (10 lop: 5 PC1 thấp nhất + 5 cao nhất)")
ax.legend(fontsize=8, loc='best', ncol=2)
plt.tight_layout()
plt.show()

### Kiểm định: Các lớp có phân tách rõ trên PC1?

In [ ]:
groups_pc1 = [X_pca_transformed[y_pca == c, 0] for c in classes]

# Normality check (5 lớp mẫu)
print("Normality check (Shapiro-Wilk tren 5 lop dau):")
for cls in classes[:5]:
    vals = X_pca_transformed[y_pca == cls, 0]
    w, p = stats.shapiro(vals)
    print(f"  {cls}: W={w:.4f}, p={p:.4f} -> {'Normal' if p > 0.05 else 'KHÔNG normal'}")

# Levene test
lev_s, lev_p = stats.levene(*groups_pc1)
print(f"\nLevene test: stat={lev_s:.2f}, p={lev_p:.2e} -> Variance {'KHÔNG đồng nhất' if lev_p < 0.05 else 'đồng nhất'}")

# ANOVA
f_val, p_val = stats.f_oneway(*groups_pc1)
print(f"\nANOVA (PC1 ~ Class): F={f_val:.2f}, p={p_val:.2e} -> {'BÁC BỎ H0' if p_val < 0.05 else 'KHÔNG BÁC BỎ H0'}")

# Kruskal-Wallis
h_val, kw_p = stats.kruskal(*groups_pc1)
print(f"Kruskal-Wallis: H={h_val:.2f}, p={kw_p:.2e} -> {'BÁC BỎ H0' if kw_p < 0.05 else 'KHÔNG BÁC BỎ H0'}")

# Eta-squared
ss_between = sum(len(g) * (g.mean() - X_pca_transformed[:, 0].mean())**2 for g in groups_pc1)
ss_total = sum((x - X_pca_transformed[:, 0].mean())**2 for x in X_pca_transformed[:, 0])
eta_sq = ss_between / ss_total
print(f"\nEta² = {eta_sq:.3f} (effect size: {'LỚN' if eta_sq>0.14 else 'TRUNG BÌNH' if eta_sq>0.06 else 'NHỎ'})")
print(f"=> Lớp ảnh giải thích {eta_sq*100:.1f}% variance cua PC1")

---
## 2. Edge Detection (Sobel & Canny)

**H0:** Edge density không khác biệt giữa 45 lop.

**H1:** Ít nhất 1 cặp lop co edge density khac biet.

### Edge Density Analysis (toàn bộ 45 lớp)

In [ ]:
EDGE_SAMPLE = 30

print("Dang tinh edge density cho toan bo 45 lop..")
edge_data = []

for cls in tqdm(classes, desc="Edge density"):
    paths = glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg"))
    np.random.seed(42)
    chosen = np.random.choice(paths, min(EDGE_SAMPLE, len(paths)), replace=False)

    for p in chosen:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)

        edges = cv2.Canny(img, 80, 160)
        density = edges.mean() / 255.0

        sobel_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
        sobel_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
        sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2).mean()

        edge_data.append({
            'class': cls,
            'canny_density': density,
            'sobel_magnitude': sobel_mag
        })

df_edge = pd.DataFrame(edge_data)
print(f"\nEdge density (Canny): mean={df_edge['canny_density'].mean():.4f}, std={df_edge['canny_density'].std():.4f}")
print(f"Sobel magnitude:     mean={df_edge['sobel_magnitude'].mean():.1f}, std={df_edge['sobel_magnitude'].std():.1f}")

In [ ]:
class_edge = df_edge.groupby('class')['canny_density'].mean().sort_values()

print("Top 5 edge density cao nhất:")
for cls, val in class_edge.tail(5).items():
    print(f"  {cls}: {val:.4f}")
print("\nTop 5 edge density thấp nhất:")
for cls, val in class_edge.head(5).items():
    print(f"  {cls}: {val:.4f}")

### Demo Edge Detection

Chọn 5 lớp đại diện từ kết quả edge density: 2 thấp nhất, 1 trung vị, 2 cao nhất.

In [ ]:
sorted_edge_classes = class_edge.index.tolist()
ne = len(sorted_edge_classes)
demo_pick = [0, 1, ne // 2, ne - 2, ne - 1]
demo_classes = [sorted_edge_classes[i] for i in demo_pick]

print("5 lop demo edge detection (chon tu sorted edge density):")
for cls in demo_classes:
    print(f"  {cls}: canny density = {class_edge[cls]:.4f}")

In [ ]:
demo_samples = load_sample(n_per_class=1, target_classes=demo_classes)

fig, axes = plt.subplots(5, 4, figsize=(14, 16))
axes[0][0].set_title("Original", fontsize=10)
axes[0][1].set_title("Sobel Combined", fontsize=10)
axes[0][2].set_title("Canny (low)", fontsize=10)
axes[0][3].set_title("Canny (high)", fontsize=10)

for row, (img, cls) in enumerate(demo_samples):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    axes[row][0].imshow(img)
    axes[row][0].set_ylabel(cls, fontsize=10, rotation=0, labelpad=60)
    axes[row][0].axis('off')

    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel_combined = np.sqrt(sobel_x**2 + sobel_y**2)
    sobel_combined = np.clip(sobel_combined / sobel_combined.max() * 255, 0, 255).astype(np.uint8)
    axes[row][1].imshow(sobel_combined, cmap='gray')
    axes[row][1].axis('off')

    canny_low = cv2.Canny(gray, 50, 100)
    axes[row][2].imshow(canny_low, cmap='gray')
    axes[row][2].axis('off')

    canny_high = cv2.Canny(gray, 100, 200)
    axes[row][3].imshow(canny_high, cmap='gray')
    axes[row][3].axis('off')

plt.suptitle("Edge Detection: Sobel vs Canny (chon tu sorted edge density)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
top_bottom_edge = list(class_edge.index[:7]) + list(class_edge.index[-7:])
df_edge_sub = df_edge[df_edge['class'].isin(top_bottom_edge)]
order_edge = class_edge[class_edge.index.isin(top_bottom_edge)].index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(data=df_edge_sub, x='class', y='canny_density', order=order_edge,
            ax=axes[0], palette='RdYlGn_r')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=8)
axes[0].set_title("Canny Edge Density (7 thấp nhất + 7 cao nhất)")
axes[0].set_ylabel("Edge Density")

class_sobel = df_edge.groupby('class')['sobel_magnitude'].mean().sort_values()
tb_sobel = list(class_sobel.index[:7]) + list(class_sobel.index[-7:])
df_sobel_sub = df_edge[df_edge['class'].isin(tb_sobel)]
order_sobel = class_sobel[class_sobel.index.isin(tb_sobel)].index.tolist()
sns.boxplot(data=df_sobel_sub, x='class', y='sobel_magnitude', order=order_sobel,
            ax=axes[1], palette='RdYlGn_r')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
axes[1].set_title("Sobel Magnitude (7 thấp nhất + 7 cao nhất)")
axes[1].set_ylabel("Mean Sobel Magnitude")

plt.tight_layout()
plt.show()

### Kiểm định: Edge density & Correlation

In [ ]:
# ANOVA + effect size
edge_groups = [df_edge[df_edge['class']==c]['canny_density'].values for c in classes]

lev_s, lev_p = stats.levene(*edge_groups)
print(f"Levene test: stat={lev_s:.2f}, p={lev_p:.2e} -> Variance {'KHÔNG đồng nhất' if lev_p < 0.05 else 'đồng nhất'}")

f_edge, p_edge = stats.f_oneway(*edge_groups)
print(f"ANOVA (Canny density ~ Class): F={f_edge:.2f}, p={p_edge:.2e} -> {'BÁC BỎ H0' if p_edge < 0.05 else 'KHÔNG BÁC BỎ H0'}")

h_edge, kw_p_edge = stats.kruskal(*edge_groups)
print(f"Kruskal-Wallis: H={h_edge:.2f}, p={kw_p_edge:.2e} -> {'BÁC BỎ H0' if kw_p_edge < 0.05 else 'KHÔNG BÁC BỎ H0'}")

ss_b = sum(len(g)*(g.mean() - df_edge['canny_density'].mean())**2 for g in edge_groups)
ss_t = sum((x - df_edge['canny_density'].mean())**2 for g in edge_groups for x in g)
eta2 = ss_b / ss_t
print(f"Eta² = {eta2:.3f} (effect size: {'LỚN' if eta2>0.14 else 'TRUNG BÌNH' if eta2>0.06 else 'NHỎ'})")

In [ ]:
# Correlation: Pearson (parametric) + Spearman (non-parametric)
r_pearson, p_pearson = stats.pearsonr(df_edge['canny_density'], df_edge['sobel_magnitude'])
r_spearman, p_spearman = stats.spearmanr(df_edge['canny_density'], df_edge['sobel_magnitude'])

print(f"Pearson  (linear):    r={r_pearson:.4f}, p={p_pearson:.2e}")
print(f"Spearman (monotonic): rho={r_spearman:.4f}, p={p_spearman:.2e}")
print(f"\n=> Cả 2 đều {'tương quan MẠNH' if abs(r_pearson) > 0.7 else 'tương quan VỪA' if abs(r_pearson) > 0.4 else 'tương quan YẾU'}")
print(f"   Sobel và Canny đo cùng 1 hiện tượng (edge/gradient) bằng cách khác nhau")

In [ ]:
class_edge_stats = df_edge.groupby('class').agg({'canny_density': 'mean', 'sobel_magnitude': 'mean'}).reset_index()

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(class_edge_stats['canny_density'], class_edge_stats['sobel_magnitude'], s=50, alpha=0.7)
for _, row in class_edge_stats.iterrows():
    ax.annotate(row['class'], (row['canny_density'], row['sobel_magnitude']), fontsize=6, alpha=0.8)
ax.set_xlabel("Canny Edge Density")
ax.set_ylabel("Mean Sobel Magnitude")
ax.set_title("Canny Density vs Sobel Magnitude theo lop")

z = np.polyfit(class_edge_stats['canny_density'], class_edge_stats['sobel_magnitude'], 1)
xline = np.linspace(class_edge_stats['canny_density'].min(), class_edge_stats['canny_density'].max(), 100)
ax.plot(xline, np.polyval(z, xline), 'r--', alpha=0.5, label=f'Pearson r={r_pearson:.3f}\nSpearman rho={r_spearman:.3f}')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Tổng kết Phân tích nâng cao

In [ ]:
print("=" * 60)
print("TONG KET PHAN TICH NANG CAO")
print("=" * 60)
print()
print("1. PCA:")
print(f"   {n95} components giữ 95% variance (tu {PCA_SIZE*PCA_SIZE} features)")
print(f"   ANOVA PC1: F={f_val:.2f}, p={p_val:.2e}, Eta²={eta_sq:.3f}")
print(f"   -> Các lớp PHÂN TÁCH được trên không gian PCA")
print()
print("2. EDGE DETECTION:")
print(f"   Canny density trung bình: {df_edge['canny_density'].mean():.4f}")
print(f"   ANOVA: F={f_edge:.2f}, Eta²={eta2:.3f} -> Các lớp khác biệt CÓ Ý NGHĨA")
print(f"   Pearson r={r_pearson:.3f}, Spearman rho={r_spearman:.3f}")
print(f"   -> Sobel & Canny tương quan mạnh, đo cùng gradient/edge")
print("=" * 60)